# Track 1 Pipeline Logic & Dataflow

Self-contained Colab notebook for video anomaly detection using:

- Multi-domain perceptual hashing: pHash, dHash, wHash
- Localized watermark SSIM verification
- Temporal scene fingerprinting from HSV histogram shifts

Upload your video and optional reference assets, then run the cells top to bottom.

## 1. Install Colab Runtime Dependencies

This installs packages inside the Colab VM only, not on your local machine.

In [ ]:
!pip install -q numpy opencv-python-headless

## 2. Upload Input Files

Upload:

- The video to scan: `.mp4`, `.mkv`, or `.mov`
- Optional watermark reference image
- Optional JSON reference hash database
- Optional JSON licensed scene fingerprint database

In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()
uploaded_files = list(uploaded.keys())
video_extensions = {".mp4", ".mkv", ".mov"}
uploaded_videos = [name for name in uploaded_files if Path(name).suffix.lower() in video_extensions]

if not uploaded_videos:
    raise ValueError("Upload at least one video file: .mp4, .mkv, or .mov")

MASTER_REFERENCE_VIDEO = next((name for name in uploaded_videos if "master" in name.lower() or "clean" in name.lower()), None)
VIDEO_FILE = next((name for name in uploaded_videos if name != MASTER_REFERENCE_VIDEO), uploaded_videos[0])
REFERENCE_HASHES_JSON = next((name for name in uploaded_files if "reference_hash" in name.lower() and name.lower().endswith(".json")), None)
LICENSED_FINGERPRINTS_JSON = next((name for name in uploaded_files if ("licensed" in name.lower() or "fingerprint" in name.lower()) and name.lower().endswith(".json")), None)
WATERMARK_IMAGE = next((name for name in uploaded_files if "watermark" in name.lower() and Path(name).suffix.lower() in {".png", ".jpg", ".jpeg", ".ppm"}), None)
WATERMARK_BOX = (958, 54, 250, 82) if WATERMARK_IMAGE else None
print(f"Selected video: {VIDEO_FILE}")
print(f"Master reference video: {MASTER_REFERENCE_VIDEO}")
print(f"Reference hashes: {REFERENCE_HASHES_JSON}")
print(f"Licensed fingerprints: {LICENSED_FINGERPRINTS_JSON}")
print(f"Watermark reference: {WATERMARK_IMAGE}")
print("Uploaded files:", uploaded_files)

## 3. Configure Scan Inputs

In [ ]:
# Optional JSON files. Auto-filled from uploaded example filenames when available.
REFERENCE_HASHES_JSON = globals().get("REFERENCE_HASHES_JSON")
LICENSED_FINGERPRINTS_JSON = globals().get("LICENSED_FINGERPRINTS_JSON")
MASTER_REFERENCE_VIDEO = globals().get("MASTER_REFERENCE_VIDEO")

# Optional watermark check. Auto-filled for watermark_reference.png when uploaded.
WATERMARK_IMAGE = globals().get("WATERMARK_IMAGE")
WATERMARK_BOX = globals().get("WATERMARK_BOX")  # x, y, width, height

# Thresholds.
FPS_STEP = 1.0
HASH_THRESHOLD_BITS = 8
PLAGIARISM_STRICT_AVG_DISTANCE = 4.0
SSIM_THRESHOLD = 0.75
SCENE_CUT_THRESHOLD = 0.45
MIN_TEMPORAL_MATCH_SEC = 5.0
SCENE_DURATION_TOLERANCE_SEC = 1.0
USE_VIRTUAL_LICENSED_TARGET = True

## 4. Optional Example Reference Schemas

Use these shapes when building real databases.

In [ ]:
example_reference_hashes = {
    "frames": [
        {
            "asset_id": "licensed_asset_001",
            "timestamp_sec": 12,
            "phash": "ffeeddccbbaa9988",
            "dhash": "0123456789abcdef",
            "whash": "89abcdef01234567",
        }
    ]
}

example_licensed_fingerprints = {
    "clips": [
        {
            "clip_id": "master_clip_001",
            "scenes": [
                {"duration_sec": 3.0, "phash": "ffeeddccbbaa9988"},
                {"duration_sec": 4.0, "phash": "0123456789abcdef"},
            ],
        }
    ]
}

## 5. Pipeline Implementation

In [ ]:
import json
import math
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import cv2
import numpy as np

HASH_BITS = 64


@dataclass
class FrameSample:
    index: int
    timestamp_sec: float
    bgr: np.ndarray
    phash: str
    dhash: str
    whash: str
    hsv_hist: np.ndarray


def bits_to_hex(bits: np.ndarray) -> str:
    bit_string = "".join("1" if bit else "0" for bit in bits.astype(bool).flatten())
    return f"{int(bit_string, 2):0{HASH_BITS // 4}x}"


def hex_to_bits(value: str) -> np.ndarray:
    value = value.strip().lower().removeprefix("0x")
    bit_string = f"{int(value, 16):0{HASH_BITS}b}"
    return np.fromiter((char == "1" for char in bit_string), dtype=np.bool_)


def hamming_distance(left: str, right: str) -> int:
    return int(np.count_nonzero(hex_to_bits(left) ^ hex_to_bits(right)))


def grayscale(frame: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)


def phash(frame: np.ndarray) -> str:
    gray = cv2.resize(grayscale(frame), (32, 32), interpolation=cv2.INTER_AREA)
    dct = cv2.dct(np.float32(gray))
    low_freq = dct[:8, :8]
    median = np.median(low_freq[1:, 1:])
    return bits_to_hex(low_freq > median)


def dhash(frame: np.ndarray) -> str:
    gray = cv2.resize(grayscale(frame), (9, 8), interpolation=cv2.INTER_AREA)
    return bits_to_hex(gray[:, 1:] > gray[:, :-1])


def whash(frame: np.ndarray) -> str:
    gray = cv2.resize(grayscale(frame), (16, 16), interpolation=cv2.INTER_AREA).astype(np.float32)
    rows = (gray[:, 0::2] + gray[:, 1::2]) / 2.0
    approx = (rows[0::2, :] + rows[1::2, :]) / 2.0
    median = np.median(approx)
    return bits_to_hex(approx > median)


def hsv_histogram(frame: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [32, 32], [0, 180, 0, 256])
    cv2.normalize(hist, hist, alpha=1.0, norm_type=cv2.NORM_L1)
    return hist.flatten()


def chi_square_distance(left: np.ndarray, right: np.ndarray) -> float:
    denom = left + right + 1e-10
    return float(0.5 * np.sum(((left - right) ** 2) / denom))


def sample_video(video_path: str, fps_step: float = 1.0) -> list[FrameSample]:
    capture = cv2.VideoCapture(video_path)
    if not capture.isOpened():
        raise ValueError(f"Could not open video file: {video_path}")

    source_fps = capture.get(cv2.CAP_PROP_FPS) or 30.0
    frame_interval = max(1, int(round(source_fps * fps_step)))
    samples = []
    frame_index = 0

    while True:
        ok, frame = capture.read()
        if not ok:
            break
        if frame_index % frame_interval == 0:
            samples.append(
                FrameSample(
                    index=frame_index,
                    timestamp_sec=frame_index / source_fps,
                    bgr=frame,
                    phash=phash(frame),
                    dhash=dhash(frame),
                    whash=whash(frame),
                    hsv_hist=hsv_histogram(frame),
                )
            )
        frame_index += 1

    capture.release()
    return samples


def load_json_or_default(path: str | None, default: Any) -> Any:
    if not path:
        return default
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def iter_reference_hashes(reference_db: dict[str, Any]) -> Iterable[dict[str, str]]:
    for item in reference_db.get("frames", []):
        if isinstance(item, dict):
            yield item


def evaluate_multihash(samples, reference_db, threshold_bits):
    refs = list(iter_reference_hashes(reference_db))
    if not refs or not samples:
        return False, None

    best_phash_distances = []
    any_match = False
    for sample in samples:
        best_phash = math.inf
        for ref in refs:
            distances = []
            for key in ("phash", "dhash", "whash"):
                if key in ref:
                    distances.append(hamming_distance(getattr(sample, key), ref[key]))
            if distances and all(distance < threshold_bits for distance in distances):
                any_match = True
            if "phash" in ref:
                best_phash = min(best_phash, hamming_distance(sample.phash, ref["phash"]))
        if best_phash < math.inf:
            best_phash_distances.append(int(best_phash))

    average = float(np.mean(best_phash_distances)) if best_phash_distances else None
    return any_match, average


def ssim(left: np.ndarray, right: np.ndarray) -> float:
    left = left.astype(np.float64)
    right = right.astype(np.float64)
    c1 = (0.01 * 255) ** 2
    c2 = (0.03 * 255) ** 2
    mu_left = left.mean()
    mu_right = right.mean()
    sigma_left = left.var()
    sigma_right = right.var()
    sigma_cross = ((left - mu_left) * (right - mu_right)).mean()
    numerator = (2 * mu_left * mu_right + c1) * (2 * sigma_cross + c2)
    denominator = (mu_left**2 + mu_right**2 + c1) * (sigma_left + sigma_right + c2)
    return float(numerator / denominator)


def evaluate_watermark(samples, watermark_path, box, threshold):
    if not samples or not watermark_path or box is None:
        return False, None
    reference = cv2.imread(watermark_path, cv2.IMREAD_GRAYSCALE)
    if reference is None:
        raise ValueError(f"Could not open watermark reference: {watermark_path}")

    x, y, width, height = box
    scores = []
    for sample in samples:
        patch = grayscale(sample.bgr)[y : y + height, x : x + width]
        if patch.size == 0:
            continue
        ref_resized = cv2.resize(reference, (patch.shape[1], patch.shape[0]), interpolation=cv2.INTER_AREA)
        scores.append(ssim(patch, ref_resized))

    if not scores:
        return False, None
    average = float(np.mean(scores))
    return average < threshold, average


def detect_scene_cuts(samples, threshold):
    cuts = []
    for index in range(1, len(samples)):
        distance = chi_square_distance(samples[index - 1].hsv_hist, samples[index].hsv_hist)
        if distance >= threshold:
            cuts.append(index)
    return cuts


def build_scene_fingerprint(samples, cuts):
    if not samples:
        return []
    boundaries = [0, *cuts, len(samples)]
    scenes = []
    for start, end in zip(boundaries, boundaries[1:]):
        if start >= end:
            continue
        scene_samples = samples[start:end]
        duration = scene_samples[-1].timestamp_sec - scene_samples[0].timestamp_sec + 1.0
        midpoint = scene_samples[len(scene_samples) // 2]
        scenes.append({"duration_sec": duration, "phash": midpoint.phash, "start_sec": scene_samples[0].timestamp_sec})
    return scenes


def build_reference_hashes_from_samples(samples):
    return {
        "frames": [
            {
                "asset_id": "auto_master_reference",
                "timestamp_sec": sample.timestamp_sec,
                "phash": sample.phash,
                "dhash": sample.dhash,
                "whash": sample.whash,
            }
            for sample in samples
        ]
    }


def build_licensed_fingerprints_from_samples(samples, scene_cut_threshold):
    cuts = detect_scene_cuts(samples, threshold=scene_cut_threshold)
    scenes = build_scene_fingerprint(samples, cuts)
    return {"clips": [{"clip_id": "auto_master_reference", "scenes": scenes}]}


def extract_watermark_reference_from_samples(samples, box):
    if not samples or box is None:
        return None
    x, y, width, height = box
    patch = grayscale(samples[0].bgr)[y : y + height, x : x + width]
    if patch.size == 0:
        return None
    path = "auto_watermark_reference.png"
    cv2.imwrite(path, patch)
    return path




def draw_virtual_watermark(frame):
    x, y, w, h = (958, 54, 250, 82)
    cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 255), -1)
    cv2.rectangle(frame, (x + 16, y + 18), (x + 55, y + 58), (35, 75, 160), -1)
    cv2.rectangle(frame, (x + 70, y + 18), (x + 109, y + 58), (35, 75, 160), -1)
    cv2.rectangle(frame, (x + 124, y + 18), (x + 163, y + 58), (35, 75, 160), -1)
    cv2.putText(frame, "LICENSED", (x + 18, y + 104), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)


def make_virtual_master_frame(sample_second):
    width, height = 1280, 720
    fps = 24
    i = int(sample_second * fps)
    scene = i // (fps * 2)
    backgrounds = [
        (40, 90, 135),
        (46, 145, 105),
        (170, 88, 54),
        (80, 70, 145),
    ]
    frame = np.full((height, width, 3), backgrounds[scene % len(backgrounds)], dtype=np.uint8)
    x = int(72 + (width - 300) * ((i % (fps * 2)) / (fps * 2)))
    y = int(205 + 50 * math.sin(i / 9))
    cv2.rectangle(frame, (x, y), (x + 180, y + 105), (245, 245, 245), -1)
    cv2.rectangle(frame, (x, y), (x + 180, y + 16), (25, 25, 25), -1)
    cv2.circle(frame, (width - 210 - (x // 5), 390), 62, (30, 35, 45), -1)
    cv2.circle(frame, (width - 210 - (x // 5), 390), 34, (245, 203, 70), -1)
    cv2.rectangle(frame, (58, 54), (335, 128), (255, 255, 255), -1)
    cv2.putText(frame, f"SCENE {scene + 1}", (78, 103), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (35, 35, 35), 3)
    cv2.putText(frame, "NORMAL: LICENSED MASTER", (54, 662), cv2.FONT_HERSHEY_SIMPLEX, 0.78, (255, 255, 255), 2)
    draw_virtual_watermark(frame)
    return frame


def make_virtual_master_samples(duration_seconds=8):
    samples = []
    for second in range(duration_seconds):
        frame = make_virtual_master_frame(second)
        samples.append(
            FrameSample(
                index=second,
                timestamp_sec=float(second),
                bgr=frame,
                phash=phash(frame),
                dhash=dhash(frame),
                whash=whash(frame),
                hsv_hist=hsv_histogram(frame),
            )
        )
    return samples

def scene_matches(query, licensed, hash_threshold, duration_tolerance):
    matched_duration = 0.0
    for query_scene, licensed_scene in zip(query, licensed):
        duration_ok = abs(query_scene["duration_sec"] - licensed_scene["duration_sec"]) <= duration_tolerance
        hash_ok = hamming_distance(query_scene["phash"], licensed_scene["phash"]) <= hash_threshold
        if not (duration_ok and hash_ok):
            break
        matched_duration += float(query_scene["duration_sec"])
    return matched_duration


def evaluate_temporal(scenes, licensed_db, min_window_sec, hash_threshold, duration_tolerance):
    for clip in licensed_db.get("clips", []):
        licensed_scenes = clip.get("scenes", [])
        for query_start in range(len(scenes)):
            for licensed_start in range(len(licensed_scenes)):
                matched_duration = scene_matches(
                    scenes[query_start:],
                    licensed_scenes[licensed_start:],
                    hash_threshold,
                    duration_tolerance,
                )
                if matched_duration >= min_window_sec:
                    return True
    return False

## 6. Run Track 1 Scan

In [ ]:
def run_track_1_scan():
    start = time.perf_counter()
    samples = sample_video(VIDEO_FILE, fps_step=FPS_STEP)
    master_samples = sample_video(MASTER_REFERENCE_VIDEO, fps_step=FPS_STEP) if MASTER_REFERENCE_VIDEO else []
    used_virtual_target = False
    if not master_samples and USE_VIRTUAL_LICENSED_TARGET:
        master_samples = make_virtual_master_samples(duration_seconds=max(8, len(samples)))
        used_virtual_target = True

    reference_hashes = load_json_or_default(REFERENCE_HASHES_JSON, {"frames": []})
    licensed_fingerprints = load_json_or_default(LICENSED_FINGERPRINTS_JSON, {"clips": []})
    watermark_image = WATERMARK_IMAGE
    watermark_box = WATERMARK_BOX

    if master_samples and not reference_hashes.get("frames"):
        reference_hashes = build_reference_hashes_from_samples(master_samples)
    if master_samples and not licensed_fingerprints.get("clips"):
        licensed_fingerprints = build_licensed_fingerprints_from_samples(master_samples, SCENE_CUT_THRESHOLD)
    if master_samples and not watermark_image:
        watermark_box = watermark_box or (958, 54, 250, 82)
        watermark_image = extract_watermark_reference_from_samples(master_samples, watermark_box)

    plagiarism_hash_match, average_phash_distance = evaluate_multihash(
        samples,
        reference_hashes,
        threshold_bits=HASH_THRESHOLD_BITS,
    )
    watermark_tampering_detected, average_ssim = evaluate_watermark(
        samples,
        watermark_image,
        watermark_box,
        threshold=SSIM_THRESHOLD,
    )
    cuts = detect_scene_cuts(samples, threshold=SCENE_CUT_THRESHOLD)
    scenes = build_scene_fingerprint(samples, cuts)
    temporal_sequence_match = evaluate_temporal(
        scenes,
        licensed_fingerprints,
        min_window_sec=MIN_TEMPORAL_MATCH_SEC,
        hash_threshold=HASH_THRESHOLD_BITS,
        duration_tolerance=SCENE_DURATION_TOLERANCE_SEC,
    )
    plagiarism_detected = (
        plagiarism_hash_match
        and average_phash_distance is not None
        and average_phash_distance <= PLAGIARISM_STRICT_AVG_DISTANCE
        and not watermark_tampering_detected
    )
    # Classification priority: watermark removal > direct frame plagiarism >
    # re-encoding. A watermark-removal attempt is often visually close to the
    # master, so do not double-label it as plagiarism.
    reencoding_detected = temporal_sequence_match and not plagiarism_detected and not watermark_tampering_detected

    latency_ms = (time.perf_counter() - start) * 1000
    anomaly = plagiarism_detected or watermark_tampering_detected or reencoding_detected

    configuration_warnings = []
    if used_virtual_target:
        configuration_warnings.append("No master/reference assets uploaded; built references from the synthetic virtual licensed target.")
    if not REFERENCE_HASHES_JSON and master_samples and not used_virtual_target:
        configuration_warnings.append("No reference hash JSON uploaded; built plagiarism reference hashes from master video.")
    elif not REFERENCE_HASHES_JSON:
        configuration_warnings.append("No reference hash JSON or master video uploaded; plagiarism detection was skipped.")
    if (not WATERMARK_IMAGE or WATERMARK_BOX is None) and master_samples and watermark_image and not used_virtual_target:
        configuration_warnings.append("No watermark image uploaded; extracted watermark reference from master video.")
    elif not watermark_image or watermark_box is None:
        configuration_warnings.append("No watermark reference image/box or master video configured; watermark tampering detection was skipped.")
    if not LICENSED_FINGERPRINTS_JSON and master_samples and not used_virtual_target:
        configuration_warnings.append("No licensed fingerprint JSON uploaded; built temporal fingerprint from master video.")
    elif not LICENSED_FINGERPRINTS_JSON:
        configuration_warnings.append("No licensed fingerprint JSON or master video uploaded; re-encoding detection was skipped.")

    return {
        "status": "anomaly_flagged" if anomaly else "clean",
        "scanned_file": Path(VIDEO_FILE).name,
        "detections": {
            "plagiarism": plagiarism_detected,
            "watermark_removal": watermark_tampering_detected,
            "reencoding": reencoding_detected,
        },
        "analytical_metrics": {
            "total_frames_sampled": len(samples),
            "average_phash_hamming_distance": average_phash_distance,
            "plagiarism_hash_match_raw": plagiarism_hash_match,
            "average_watermark_ssim_score": average_ssim,
            "detected_scene_cuts_count": len(cuts),
            "temporal_sequence_match": temporal_sequence_match,
            "processing_latency_ms": round(latency_ms, 1),
        },
        "configuration_warnings": configuration_warnings,
        "debug_artifacts": {
            "scene_fingerprint": scenes,
            "sample_hashes": [
                {
                    "timestamp_sec": sample.timestamp_sec,
                    "phash": sample.phash,
                    "dhash": sample.dhash,
                    "whash": sample.whash,
                }
                for sample in samples
            ],
        },
    }


track_1_result = run_track_1_scan()
print(json.dumps(track_1_result, indent=2))